# Physical AI Platform - Hosted Compute and Accelerator Workbench

**Iteration 50 - browser-first execution through Google Colab or a compatible hosted Jupyter runtime**

**Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.**  
**AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.**  
**Project licence: GNU General Public License v3.0 only (GPL-3.0-only).** See the repository `LICENSE` file and <https://www.gnu.org/licenses/gpl-3.0.html>.

This notebook is a platform-owned, reproducible numerical-compute adapter and validation harness. It is not a new canonical model, does not replace the platform kernel, and does not make a notebook or GPU runtime the source of truth. It uses mature numerical libraries as replaceable execution backends and emits a configuration-bound, hash-bearing **SIMULATION** result envelope for import by the platform.

## Open this notebook in Colab

1. Open <https://colab.research.google.com/> in the browser.
2. Choose **File → Upload notebook** and select this `.ipynb` file from the supplied ZIP, or open the notebook from the published repository using Colab's GitHub tab.
3. For GPU execution choose **Runtime → Change runtime type → T4 GPU** (or another available accelerator). Availability is controlled by Google and is not guaranteed.
4. Run all cells from the top. The notebook probes the actual runtime; it does not assume a GPU merely because a GPU runtime was requested.

When the platform repository has a stable public GitHub owner/name and branch, Grok should replace the generic upload workflow with this direct deep link:

`https://colab.research.google.com/github/<OWNER>/<REPOSITORY>/blob/<BRANCH>/notebooks/PAP_Hosted_Compute_Backends.ipynb`

Do not publish a fabricated owner/repository link. The current supplied handoff does not identify the platform repository's GitHub slug.

## Copyright, licensing and upstreams

The notebook's original orchestration, validation, test harness and platform adapter material are GPL-3.0-only under the copyright notice above. Third-party packages retain their own licences; installing or using them does not relicense them. Record exact installed versions and consult each upstream's current licence and notices before redistributing binaries, wheels, containers or modified source. The notebook does not vendor upstream source.

Primary upstreams covered by the platform's backend catalogue include NumPy, SciPy, PyTorch, JAX, CuPy, Taichi, NVIDIA Warp, PETSc, FEniCSx, MFEM, SU2, OpenFOAM, PyBullet, MuJoCo, Project Chrono and Elmer FEM. Their installation and acceleration capabilities differ. Some are pip-friendly; others require native builds, MPI, containers or an external solver runner. The catalogue is a discovery/adapter map, not a claim that all packages are installed in every Colab image.

## Safety and assurance boundary

- A notebook run is **SIMULATION**, never HIL or physical verification.
- A successful GPU kernel launch is not solver validation.
- A converged residual is not sufficient by itself: compare to a manufactured/analytical solution and perform grid-refinement checks.
- Colab is ephemeral and resource-limited. Persist the notebook and result artifact explicitly; do not assume the VM survives disconnection.
- Do not put credentials, private keys, personal data or safety-critical actuation secrets in notebook cells or outputs.
- Do not connect this notebook directly to actuators. HIL uses the separately governed HIL module and hardware commissioning path.

## 0. Runtime policy

The reference path is CPU-capable and reproducible. Optional GPU frameworks are probed independently. No package is installed automatically beyond what the hosted runtime already provides. If an optional framework is missing, that backend is marked unavailable rather than silently substituted.

Set `REQUESTED_BACKEND_ID` and `REQUESTED_ACCELERATOR` in the first configuration cell before launch. These values must match the platform execution plan. The default is the SciPy CPU reference. Choose `pytorch` + `nvidia-gpu` or `jax` + `nvidia-gpu` to request an actual GPU path. If the requested backend/device is absent, the notebook records a failed check; it does not silently fall back to CPU.

The notebook also includes optional executable adapters for Numba matrix-free CG, CuPy sparse CG and PyAMG-preconditioned SciPy CG. Select one of these explicitly in `REQUESTED_BACKEND_ID` and install it by adding its key to `OPTIONAL_PIP_INSTALLS`.

Taichi and NVIDIA Warp also have executable CPU/GPU kernel parity probes. They are kernel frameworks, not pre-validated CFD/FEM solvers. The optional package install cell installs only explicitly selected packages and does not modify the base Colab environment unless the user opts in.

In [ ]:
from __future__ import annotations
import os, sys, json, time, math, hashlib, platform, importlib.util, subprocess, resource
from datetime import datetime, timezone
from pathlib import Path

PROJECT_ID = "PAP-HOSTED-COMPUTE-DEMO"       # Replace with canonical project ID
CONFIGURATION_ID = "CFG-MMS-POISSON-001"    # Replace with canonical configuration/revision ID
RUN_ID = "RUN-ITER50-MMS-001"                # Must be unique within the project/configuration
REQUESTED_BACKEND_ID = "numpy-scipy"          # numpy-scipy | pytorch | jax | numba | cupy | pyamg
REQUESTED_ACCELERATOR = "cpu"                 # cpu | nvidia-gpu | tpu (JAX supports TPU)
OPTIONAL_PIP_INSTALLS = []                    # Explicit opt-in: e.g. ["cupy", "warp", "taichi"]
REQUIREMENT_IDS = ["REQ-NUMERICAL-VERIFICATION", "REQ-TRACEABILITY"]
GRID_SIZES = (16, 32, 64)                    # Interior nodes per axis
MAX_RELATIVE_L2_ERROR = 2.0e-3               # Acceptance bound for the selected grid sequence
MAX_RELATIVE_RESIDUAL = 1.0e-9

runtime = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "colab": bool("google.colab" in sys.modules or importlib.util.find_spec("google.colab")),
    "started_utc": datetime.now(timezone.utc).isoformat(),
}
print(json.dumps(runtime, indent=2))


In [ ]:
# Optional package activation. Nothing installs unless the user edits the list above.
# Install one GPU stack at a time; conflicting CUDA/XLA/native wheels may require a runtime restart.
OPTIONAL_PACKAGE_MAP = {
    "cupy": "cupy",                 # resolved to cupy-cuda12x[ctk]/cupy-cuda13x[ctk] below
    "jax-cuda": "jax",               # resolved to jax[cuda12]/jax[cuda13] below
    "warp": "warp-lang",
    "taichi": "taichi",
    "pyamg": "pyamg",
    "numba": "numba",
    "scikit-fem": "scikit-fem",
    "torchdiffeq": "torchdiffeq",
    "diffrax": "diffrax",
    "meshio": "meshio",
    "pyvista": "pyvista",
    "pybullet": "pybullet",
    "mujoco": "mujoco",
}
if OPTIONAL_PIP_INSTALLS:
    unknown = sorted(set(OPTIONAL_PIP_INSTALLS) - set(OPTIONAL_PACKAGE_MAP))
    if unknown:
        raise ValueError(f"Unknown optional package keys: {unknown}")
    for package_key in OPTIONAL_PIP_INSTALLS:
        package_spec = OPTIONAL_PACKAGE_MAP[package_key]
        if package_key in ("cupy", "jax-cuda"):
            cuda_text = ""
            try:
                probe = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
                cuda_text = probe.stdout + probe.stderr
            except Exception:
                pass
            import re
            match = re.search(r"CUDA Version:\s*(\d+)", cuda_text)
            if not match:
                raise RuntimeError("Cannot determine CUDA major version; do not guess a CuPy wheel.")
            if package_key == "cupy":
                package_spec = "cupy-cuda13x[ctk]" if int(match.group(1)) >= 13 else "cupy-cuda12x[ctk]"
            else:
                package_spec = "jax[cuda13]" if int(match.group(1)) >= 13 else "jax[cuda12]"
        print("Installing explicitly requested optional backend:", package_key, package_spec)
        subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", package_spec], check=True)
    print("Optional installs finished. If a native/CUDA library requests restart, restart the runtime and run all cells again.")
else:
    print("No optional package installation requested. Existing runtime packages will be probed as-is.")

In [ ]:
# Probe installed numerical runtimes. Probe is informational; no GPU is assumed.
backend_probe = {}
for name in ("numpy", "scipy", "torch", "jax", "cupy", "taichi", "warp", "numba", "pyamg", "skfem", "torchdiffeq", "diffrax", "meshio", "pyvista", "pybullet", "mujoco", "petsc4py", "dolfinx"):
    present = importlib.util.find_spec(name) is not None
    item = {"installed": present, "version": None, "devices": []}
    if present:
        try:
            mod = __import__(name)
            item["version"] = getattr(mod, "__version__", "unknown")
            if name == "torch":
                item["devices"] = (["cuda:" + str(i) for i in range(mod.cuda.device_count())] if mod.cuda.is_available() else ["cpu"])
            elif name == "jax":
                item["devices"] = [str(d) for d in mod.devices()]
            elif name == "cupy":
                try: item["devices"] = ["cuda:" + str(mod.cuda.runtime.getDevice())]
                except Exception: item["devices"] = []
        except Exception as exc:
            item["probe_error"] = f"{type(exc).__name__}: {exc}"
    backend_probe[name] = item
print(json.dumps(backend_probe, indent=2))


## 1. Reference engineering case - manufactured-solution Poisson problem

Solve the two-dimensional Dirichlet problem on the unit square:

\[ -\nabla^2 u = 2\pi^2\sin(\pi x)\sin(\pi y), \qquad u|_{\partial\Omega}=0. \]

The exact solution is \(u=\sin(\pi x)\sin(\pi y)\). The five-point second-order finite-difference stencil is assembled as a sparse matrix and solved using SciPy Conjugate Gradient (CG). This provides a CPU reference implementation with an analytical answer. The grid-refinement test checks the expected second-order trend; the residual and analytical error are reported separately. This is a validation harness for a bounded benchmark, not a general-purpose CFD/FEM solver.

In [ ]:
import numpy as np
import scipy
from scipy.sparse import diags, eye, kron
from scipy.sparse.linalg import cg


def poisson_system(n: int):
    """Return CSR matrix and RHS for -Laplacian on n x n interior nodes."""
    if not isinstance(n, int) or n < 4 or n > 512:
        raise ValueError("n must be an integer in [4, 512]")
    h = 1.0 / (n + 1)
    x = np.arange(1, n + 1, dtype=np.float64) * h
    X, Y = np.meshgrid(x, x, indexing="ij")
    exact = np.sin(np.pi * X) * np.sin(np.pi * Y)
    forcing = 2.0 * np.pi**2 * exact
    T = diags([-np.ones(n-1), 2.0*np.ones(n), -np.ones(n-1)], [-1, 0, 1], format="csr")
    A = (kron(eye(n, format="csr"), T, format="csr") + kron(T, eye(n, format="csr"), format="csr"))
    b = (h*h*forcing).reshape(-1)
    return A, b, exact, h


def solve_scipy(n: int, rtol: float = 1e-11):
    A, b, exact, h = poisson_system(n)
    started = time.perf_counter()
    iterations = [0]
    def count(_): iterations[0] += 1
    x, info = cg(A, b, rtol=rtol, atol=0.0, maxiter=20*n*n, callback=count)
    elapsed = time.perf_counter() - started
    if info != 0:
        raise RuntimeError(f"CG did not converge: info={info}")
    u = x.reshape(n, n)
    rel_l2 = float(np.linalg.norm(u-exact) / np.linalg.norm(exact))
    rel_residual = float(np.linalg.norm(A @ x-b) / np.linalg.norm(b))
    return {"n":n, "h":h, "relative_l2_error":rel_l2, "relative_residual":rel_residual,
            "iterations":iterations[0], "wall_seconds":elapsed, "solution":u}

reference_runs = [solve_scipy(n) for n in GRID_SIZES]
for row in reference_runs:
    print({k:v for k,v in row.items() if k != "solution"})


In [ ]:
# Falsifiable boundary checks for the discrete model builder.
for invalid_n in (3, 513, 4.0, True):
    try:
        poisson_system(invalid_n)
        raise AssertionError(f"invalid grid size accepted: {invalid_n!r}")
    except ValueError:
        pass
A4,b4,u4,h4 = poisson_system(4)
assert A4.shape == (16,16) and b4.shape == (16,) and np.isfinite(b4).all()
A512,b512,u512,h512 = poisson_system(512)
assert A512.shape == (512*512,512*512) and b512.shape == (512*512,)
assert np.allclose(A4.toarray(), A4.toarray().T, rtol=0, atol=0)
print("Grid input boundaries and symmetry checks: PASS")

In [ ]:
# Grid refinement: error should decrease at approximately O(h^2).
errors = [r["relative_l2_error"] for r in reference_runs]
orders = [math.log(errors[i]/errors[i+1], 2) for i in range(len(errors)-1)]
assert all(errors[i+1] < errors[i] for i in range(len(errors)-1)), f"No monotone refinement: {errors}"
assert all(order > 1.7 for order in orders), f"Observed order below expected second order: {orders}"
assert reference_runs[-1]["relative_l2_error"] < MAX_RELATIVE_L2_ERROR
assert max(r["relative_residual"] for r in reference_runs) < MAX_RELATIVE_RESIDUAL
print({"grid_errors":errors, "observed_orders":orders, "max_residual":max(r["relative_residual"] for r in reference_runs), "PASS":True})


## 2. PyTorch tensor backend - CPU/GPU matrix-free CG

This cell reuses the same discrete operator but applies it matrix-free through tensor stencil operations. The selected device is taken from the runtime probe. When Colab has actually assigned an NVIDIA GPU, this cell runs on CUDA; otherwise it runs on CPU and reports that fact. GPU use is not inferred from the notebook setting alone.

In [ ]:
torch_result = None
if importlib.util.find_spec("torch") is not None:
    import torch
    import torch.nn.functional as F
    torch_device_available = (REQUESTED_ACCELERATOR == "cpu" or (REQUESTED_ACCELERATOR == "nvidia-gpu" and torch.cuda.is_available()))
    if torch_device_available:
        device = torch.device("cuda" if REQUESTED_ACCELERATOR == "nvidia-gpu" else "cpu")
        dtype = torch.float64
        if device.type == "cuda":
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        n = 32
        h = 1.0/(n+1)
        xg = torch.arange(1,n+1,device=device,dtype=dtype)*h
        Xg,Yg = torch.meshgrid(xg,xg,indexing="ij")
        exact_t = torch.sin(torch.pi*Xg)*torch.sin(torch.pi*Yg)
        rhs = h*h*2.0*torch.pi**2*exact_t
        def apply_A(u):
            pad = F.pad(u, (1,1,1,1), mode="constant", value=0.0)
            return 4.0*u - pad[2:,1:-1] - pad[:-2,1:-1] - pad[1:-1,2:] - pad[1:-1,:-2]
        started = time.perf_counter()
        x = torch.zeros_like(rhs)
        r = rhs - apply_A(x)
        p = r.clone()
        rr = torch.sum(r*r)
        initial = torch.sqrt(rr)
        it = 0
        for it in range(1, 20*n*n+1):
            Ap = apply_A(p)
            denom = torch.sum(p*Ap)
            if not torch.isfinite(denom) or denom <= 0:
                raise RuntimeError("CG encountered non-positive/non-finite curvature")
            alpha = rr/denom
            x = x + alpha*p
            r = r - alpha*Ap
            rr_new = torch.sum(r*r)
            if torch.sqrt(rr_new) <= 1e-10*initial:
                rr = rr_new
                break
            beta = rr_new/rr
            p = r + beta*p
            rr = rr_new
        if device.type == "cuda": torch.cuda.synchronize()
        elapsed = time.perf_counter()-started
        rel_l2 = float((torch.linalg.vector_norm(x-exact_t)/torch.linalg.vector_norm(exact_t)).item())
        rel_res = float((torch.linalg.vector_norm(apply_A(x)-rhs)/torch.linalg.vector_norm(rhs)).item())
        torch_result = {"backend":"pytorch", "device":str(device), "precision":"float64", "n":n,
                        "relative_l2_error":rel_l2, "relative_residual":rel_res, "iterations":it,
                        "wall_seconds":elapsed}
        assert rel_l2 < MAX_RELATIVE_L2_ERROR, torch_result
        assert rel_res < MAX_RELATIVE_RESIDUAL, torch_result
        print(torch_result)
    else:
        torch_result = {"backend":"pytorch", "status":"REQUESTED_DEVICE_UNAVAILABLE", "device":"UNAVAILABLE"}
        print(torch_result)
else:
    torch_result = {"backend":"pytorch", "status":"NOT_INSTALLED", "device":"UNAVAILABLE"}
    print(torch_result)

## 3. Optional JAX backend

JAX/XLA is probed separately. If present, its Conjugate Gradient implementation is applied to the same matrix-free stencil. It may target CPU, NVIDIA GPU or TPU depending on the runtime's actual JAX devices and installed wheels. If the device plugin is incompatible, the failure is reported as an optional-backend failure; the SciPy reference remains independently available. Consult the current JAX installation matrix before changing wheels: <https://docs.jax.dev/en/latest/installation.html>.

In [ ]:
jax_result = None
if importlib.util.find_spec("jax") is not None:
    try:
        import jax
        jax.config.update("jax_enable_x64", True)
        import jax.numpy as jnp
        from jax.scipy.sparse.linalg import cg as jax_cg
        wanted_platform = {"cpu":"cpu", "nvidia-gpu":"gpu", "tpu":"tpu"}.get(REQUESTED_ACCELERATOR)
        selected = next((d for d in jax.devices() if d.platform == wanted_platform), None)
        if selected is None:
            jax_result = {"backend":"jax", "status":"REQUESTED_DEVICE_UNAVAILABLE", "devices":[str(d) for d in jax.devices()], "device":"UNAVAILABLE"}
        else:
            n = 32
            h = 1.0/(n+1)
            with jax.default_device(selected):
                xj = jnp.arange(1,n+1,dtype=jnp.float64)*h
                Xj,Yj = jnp.meshgrid(xj,xj,indexing="ij")
                exact_j = jnp.sin(jnp.pi*Xj)*jnp.sin(jnp.pi*Yj)
                rhs_j = h*h*2.0*jnp.pi**2*exact_j
                def apply_A_j(u):
                    pad = jnp.pad(u, ((1,1),(1,1)))
                    return 4.0*u-pad[2:,1:-1]-pad[:-2,1:-1]-pad[1:-1,2:]-pad[1:-1,:-2]
                started = time.perf_counter()
                sol_j, info_j = jax_cg(apply_A_j, rhs_j, tol=1e-10, atol=0.0, maxiter=20*n*n)
                sol_j.block_until_ready()
                elapsed = time.perf_counter()-started
                rel_l2 = float((jnp.linalg.norm(sol_j-exact_j)/jnp.linalg.norm(exact_j)).item())
                rel_res = float((jnp.linalg.norm(apply_A_j(sol_j)-rhs_j)/jnp.linalg.norm(rhs_j)).item())
            device_memory_gib = None
            try:
                stats = selected.memory_stats() or {}
                if stats.get("peak_bytes_in_use") is not None:
                    device_memory_gib = float(stats["peak_bytes_in_use"] / (1024**3))
            except Exception:
                pass
            jax_result = {"backend":"jax", "devices":[str(d) for d in jax.devices()], "device":str(selected), "devicePlatform":selected.platform, "precision":"float64",
                          "relative_l2_error":rel_l2, "relative_residual":rel_res, "wall_seconds":elapsed,
                          "peakDeviceMemoryGiB":device_memory_gib, "info":str(info_j)}
            assert rel_l2 < MAX_RELATIVE_L2_ERROR, jax_result
            assert rel_res < MAX_RELATIVE_RESIDUAL, jax_result
        print(jax_result)
    except Exception as exc:
        jax_result = {"backend":"jax", "status":"OPTIONAL_BACKEND_FAILED", "error":f"{type(exc).__name__}: {exc}"}
        print(jax_result)
else:
    jax_result = {"backend":"jax", "status":"NOT_INSTALLED"}
    print(jax_result)

## 4. Numba JIT backend - compiled CPU stencil

Numba is an optional JIT compiler for numerical Python. This case compiles the matrix-free stencil and Conjugate Gradient loop to native CPU code. The implementation is checked against the same manufactured solution and residual. It is a CPU JIT path; it is not represented as CUDA execution.

In [ ]:
numba_result = None
if importlib.util.find_spec("numba") is not None:
    import numba
    @numba.njit
    def numba_apply_A(u):
        n = u.shape[0]
        out = np.empty_like(u)
        for i in range(n):
            for j in range(n):
                value = 4.0*u[i,j]
                if i > 0: value -= u[i-1,j]
                if i+1 < n: value -= u[i+1,j]
                if j > 0: value -= u[i,j-1]
                if j+1 < n: value -= u[i,j+1]
                out[i,j] = value
        return out
    @numba.njit
    def numba_cg(b, tol, maxiter):
        x = np.zeros_like(b)
        r = b - numba_apply_A(x)
        p = r.copy()
        rr = np.sum(r*r)
        initial = np.sqrt(rr)
        it = 0
        for it in range(1, maxiter+1):
            Ap = numba_apply_A(p)
            denom = np.sum(p*Ap)
            if not np.isfinite(denom) or denom <= 0.0:
                return x, it, -1
            alpha = rr/denom
            x = x + alpha*p
            r = r - alpha*Ap
            rr_new = np.sum(r*r)
            if np.sqrt(rr_new) <= tol*initial:
                return x, it, 0
            beta = rr_new/rr
            p = r + beta*p
            rr = rr_new
        return x, it, 1
    n = 32
    h = 1.0/(n+1)
    xx = np.arange(1,n+1,dtype=np.float64)*h
    XX,YY = np.meshgrid(xx,xx,indexing="ij")
    exact_n = np.sin(np.pi*XX)*np.sin(np.pi*YY)
    rhs_n = h*h*2.0*np.pi**2*exact_n
    started = time.perf_counter()
    sol_n,it_n,info_n = numba_cg(rhs_n,1e-10,20*n*n)
    elapsed = time.perf_counter()-started
    err_n = float(np.linalg.norm(sol_n-exact_n)/np.linalg.norm(exact_n))
    res_n = float(np.linalg.norm(numba_apply_A(sol_n)-rhs_n)/np.linalg.norm(rhs_n))
    numba_result = {"backend":"numba","version":numba.__version__,"device":"CPU","precision":"float64",
                    "relative_l2_error":err_n,"relative_residual":res_n,"iterations":int(it_n),"info":int(info_n),"wall_seconds":elapsed}
    assert info_n == 0 and err_n < MAX_RELATIVE_L2_ERROR and res_n < MAX_RELATIVE_RESIDUAL, numba_result
    print(numba_result)
else:
    numba_result = {"backend":"numba","status":"NOT_INSTALLED"}
    print(numba_result)

## 5. Optional backend activation map

Do not install every library into one runtime indiscriminately. They have overlapping CUDA/XLA/native dependencies and may conflict. Enable one stack at a time, restart the runtime when its documentation requires it, and rerun the capability probe.

| Library | Typical use | Colab activation guidance | Status in this notebook |
|---|---|---|---|
| PyTorch | Tensor kernels, autodiff, ML and custom PDE prototypes | Usually present; select GPU runtime and verify `torch.cuda.is_available()` | Executed if importable |
| JAX/XLA | Compiled array programming, autodiff, optimization | Use official wheel matching actual CUDA/driver or TPU runtime | Executed if importable |
| CuPy | CUDA arrays and sparse CG | Select the CUDA-matched wheel; the notebook includes a sparse Poisson CG adapter | Executed if installed and CUDA works |
| NVIDIA Warp | Custom GPU physics kernels | Optional install; deterministic kernel parity smoke included | Executed if installed |
| Taichi | Data-oriented GPU kernels | Optional install; deterministic kernel parity smoke included | Executed if installed |
| Numba, PyAMG, scikit-fem | JIT kernels, algebraic multigrid and FEM assembly | Optional pip; Numba and PyAMG adapters included, scikit-fem is catalogued for the FEM extension | Numba executed; PyAMG if installed |
| PETSc/petsc4py, FEniCSx, MFEM | Sparse/Krylov, FEM and multiphysics | Usually needs matched MPI/native stack or container; use dedicated hosted runner for serious jobs | External/advanced runner |
| SU2, OpenFOAM, Elmer | Mature CFD/FEM executables | Prefer pinned binary/container or dedicated hosted runner; Colab setup is not guaranteed | External solver adapter path |
| PyBullet, MuJoCo, Project Chrono | Rigid-body/multibody/robotics | Python wheels for PyBullet/MuJoCo; native build for Chrono | Optional free-fall smoke for PyBullet/MuJoCo; Chrono external |

Official installation references: <https://docs.cupy.dev/en/stable/install.html>, <https://github.com/NVIDIA/warp>, <https://www.taichi-lang.org/>, <https://petsc.org/>, <https://fenicsproject.org/>, <https://mfem.org/>, <https://su2code.github.io/>, <https://openfoam.org/>.

Colab resource availability is dynamic and GPU/TPU types vary by session. For JAX, choose the current CUDA 12/13 wheel only after checking the driver requirements. For TPU, use the currently supported Colab TPU runtime (JAX documentation currently cautions against older deprecated TPU runtimes).

In [ ]:
cupy_result = None
if importlib.util.find_spec("cupy") is not None:
    try:
        import cupy as cp
        from cupyx.scipy.sparse import csr_matrix as cupy_csr
        from cupyx.scipy.sparse.linalg import cg as cupy_cg
        cp.cuda.runtime.getDevice()
        n = 32
        A_cpu,b_cpu,exact_cpu,h_cpu = poisson_system(n)
        A_gpu = cupy_csr(A_cpu)
        b_gpu = cp.asarray(b_cpu)
        exact_gpu = cp.asarray(exact_cpu.reshape(-1))
        started = time.perf_counter()
        x_gpu,info_gpu = cupy_cg(A_gpu,b_gpu,rtol=1e-10,atol=0.0,maxiter=20*n*n)
        cp.cuda.Stream.null.synchronize()
        elapsed = time.perf_counter()-started
        err_gpu = float((cp.linalg.norm(x_gpu-exact_gpu)/cp.linalg.norm(exact_gpu)).get())
        res_gpu = float((cp.linalg.norm(A_gpu@x_gpu-b_gpu)/cp.linalg.norm(b_gpu)).get())
        cupy_result = {"backend":"cupy","version":cp.__version__,"device":"cuda:"+str(cp.cuda.runtime.getDevice()),"precision":"float64",
                       "relative_l2_error":err_gpu,"relative_residual":res_gpu,"info":int(info_gpu),"wall_seconds":elapsed,
                       "array_parity":"PASS" if abs(float(cp.sum(cp.arange(1024,dtype=cp.float64)).get())-float(np.sum(np.arange(1024,dtype=np.float64)))) < 1e-9 else "FAIL"}
        assert info_gpu == 0 and err_gpu < MAX_RELATIVE_L2_ERROR and res_gpu < MAX_RELATIVE_RESIDUAL and cupy_result["array_parity"] == "PASS", cupy_result
        print(cupy_result)
    except Exception as exc:
        cupy_result = {"backend":"cupy","status":"DEVICE_OR_SOLVER_UNAVAILABLE","error":f"{type(exc).__name__}: {exc}"}
        print(cupy_result)
else:
    cupy_result = {"backend":"cupy","status":"NOT_INSTALLED"}
    print(cupy_result)

## 6. Optional GPU kernel frameworks - Taichi and NVIDIA Warp

Taichi and Warp are not general-purpose PDE solvers by themselves. They are useful ways to bring custom, application-specific numerical kernels onto CPU/GPU devices. These cells execute a deterministic vector-add kernel and verify exact output parity. For a real solver, the kernel must be paired with domain equations, boundary conditions, convergence criteria and validation cases.

In [ ]:
taichi_result = None
if importlib.util.find_spec("taichi") is not None:
    try:
        import taichi as ti
        arch = ti.cuda if REQUESTED_ACCELERATOR == "nvidia-gpu" else ti.cpu
        ti.init(arch=arch, offline_cache=False, log_level=ti.ERROR)
        @ti.kernel
        def ti_add(a: ti.types.ndarray(dtype=ti.f32, ndim=1), b: ti.types.ndarray(dtype=ti.f32, ndim=1), out: ti.types.ndarray(dtype=ti.f32, ndim=1)):
            for i in range(out.shape[0]):
                out[i] = a[i] + b[i]
        aa=np.arange(256,dtype=np.float32); bb=np.arange(256,dtype=np.float32)*2; cc=np.zeros_like(aa)
        ti_add(aa,bb,cc)
        taichi_result={"backend":"taichi","version":ti.__version__,"device":str(arch),"parity":"PASS" if np.array_equal(cc,aa+bb) else "FAIL"}
        assert taichi_result["parity"] == "PASS", taichi_result
        print(taichi_result)
    except Exception as exc:
        taichi_result={"backend":"taichi","status":"DEVICE_OR_KERNEL_UNAVAILABLE","error":f"{type(exc).__name__}: {exc}"}
        print(taichi_result)
else:
    taichi_result={"backend":"taichi","status":"NOT_INSTALLED"}
    print(taichi_result)

In [ ]:
warp_result = None
if importlib.util.find_spec("warp") is not None:
    try:
        import warp as wp
        wp.init()
        @wp.kernel
        def wp_add(a: wp.array(dtype=float), b: wp.array(dtype=float), out: wp.array(dtype=float)):
            i=wp.tid()
            out[i]=a[i]+b[i]
        aa=np.arange(256,dtype=np.float32); bb=np.arange(256,dtype=np.float32)*2
        device="cuda:0" if REQUESTED_ACCELERATOR == "nvidia-gpu" else "cpu"
        wa=wp.array(aa,dtype=wp.float32,device=device); wb=wp.array(bb,dtype=wp.float32,device=device)
        wc=wp.zeros(256,dtype=wp.float32,device=device)
        wp.launch(wp_add,dim=256,inputs=[wa,wb,wc],device=device); wp.synchronize_device(device)
        result=wc.numpy()
        warp_result={"backend":"warp","version":getattr(wp,"__version__","unknown"),"device":device,"parity":"PASS" if np.array_equal(result,aa+bb) else "FAIL"}
        assert warp_result["parity"] == "PASS", warp_result
        print(warp_result)
    except Exception as exc:
        warp_result={"backend":"warp","status":"DEVICE_OR_KERNEL_UNAVAILABLE","error":f"{type(exc).__name__}: {exc}"}
        print(warp_result)
else:
    warp_result={"backend":"warp","status":"NOT_INSTALLED"}
    print(warp_result)

## 7. Optional ODE, mesh and rigid-body library adapters

These small smoke cases make several additional upstreams directly usable from the hosted notebook when selected and installed. They are deliberately separated from the Poisson primary result: an ODE check does not validate a PDE solver, a mesh round-trip does not validate mesh quality, and a rigid-body smoke simulation does not certify a robot.

In [ ]:
optional_library_results = {}
# Differentiable ODE integration with torchdiffeq.
if importlib.util.find_spec("torchdiffeq") is not None and importlib.util.find_spec("torch") is not None:
    try:
        from torchdiffeq import odeint
        class ExponentialDecay(torch.nn.Module):
            def forward(self,t,y): return -y
        ts=torch.linspace(0.0,1.0,11,dtype=torch.float64)
        ys=odeint(ExponentialDecay(),torch.tensor([1.0],dtype=torch.float64),ts,rtol=1e-9,atol=1e-11)
        err=abs(float(ys[-1,0].item())-math.exp(-1.0))
        optional_library_results["torchdiffeq"]={"status":"PASS" if err<1e-7 else "FAIL","relative_error":err}
    except Exception as exc: optional_library_results["torchdiffeq"]={"status":"UNAVAILABLE","error":f"{type(exc).__name__}: {exc}"}
else: optional_library_results["torchdiffeq"]={"status":"NOT_INSTALLED"}

# Differentiable ODE integration with Diffrax/JAX.
if importlib.util.find_spec("diffrax") is not None and importlib.util.find_spec("jax") is not None:
    try:
        import diffrax
        term=diffrax.ODETerm(lambda t,y,args:-y)
        sol=diffrax.diffeqsolve(term,diffrax.Tsit5(),t0=0.0,t1=1.0,dt0=0.1,y0=jnp.asarray(1.0),saveat=diffrax.SaveAt(t1=True),rtol=1e-8,atol=1e-8)
        err=abs(float(sol.ys[0])-math.exp(-1.0))
        optional_library_results["diffrax"]={"status":"PASS" if err<1e-6 else "FAIL","absolute_error":err}
    except Exception as exc: optional_library_results["diffrax"]={"status":"UNAVAILABLE","error":f"{type(exc).__name__}: {exc}"}
else: optional_library_results["diffrax"]={"status":"NOT_INSTALLED"}

# Mesh I/O round-trip using meshio.
if importlib.util.find_spec("meshio") is not None:
    try:
        import meshio, tempfile
        points=np.array([[0.,0.,0.],[1.,0.,0.],[0.,1.,0.]])
        mesh=meshio.Mesh(points,[('triangle',np.array([[0,1,2]],dtype=np.int64))])
        with tempfile.NamedTemporaryFile(suffix='.vtk') as tmp:
            meshio.write(tmp.name,mesh,file_format='vtk')
            loaded=meshio.read(tmp.name)
            ok_mesh=loaded.points.shape==(3,3) and len(loaded.cells)==1 and loaded.cells[0].data.shape==(1,3)
        optional_library_results["meshio"]={"status":"PASS" if ok_mesh else "FAIL"}
    except Exception as exc: optional_library_results["meshio"]={"status":"UNAVAILABLE","error":f"{type(exc).__name__}: {exc}"}
else: optional_library_results["meshio"]={"status":"NOT_INSTALLED"}

# Mesh construction/inspection using PyVista (no display server required).
if importlib.util.find_spec("pyvista") is not None:
    try:
        import pyvista as pv
        surface=pv.PolyData(np.array([[0.,0.,0.],[1.,0.,0.],[0.,1.,0.]]),faces=np.array([3,0,1,2]))
        ok_surface=surface.n_points==3 and surface.n_cells==1 and np.allclose(surface.bounds,(0,1,0,1,0,0))
        optional_library_results["pyvista"]={"status":"PASS" if ok_surface else "FAIL","points":surface.n_points}
    except Exception as exc: optional_library_results["pyvista"]={"status":"UNAVAILABLE","error":f"{type(exc).__name__}: {exc}"}
else: optional_library_results["pyvista"]={"status":"NOT_INSTALLED"}

# Rigid-body free-fall smoke using PyBullet DIRECT mode.
if importlib.util.find_spec("pybullet") is not None:
    try:
        import pybullet as pb
        cid=pb.connect(pb.DIRECT)
        pb.setGravity(0,0,-9.81,physicsClientId=cid)
        shape=pb.createCollisionShape(pb.GEOM_BOX,halfExtents=[0.05,0.05,0.05],physicsClientId=cid)
        body=pb.createMultiBody(baseMass=1.0,baseCollisionShapeIndex=shape,basePosition=[0,0,1],physicsClientId=cid)
        for _ in range(120): pb.stepSimulation(physicsClientId=cid)
        z=float(pb.getBasePositionAndOrientation(body,physicsClientId=cid)[0][2])
        pb.disconnect(physicsClientId=cid)
        optional_library_results["pybullet"]={"status":"PASS" if 0.0<z<1.0 else "FAIL","final_z":z}
    except Exception as exc: optional_library_results["pybullet"]={"status":"UNAVAILABLE","error":f"{type(exc).__name__}: {exc}"}
else: optional_library_results["pybullet"]={"status":"NOT_INSTALLED"}

# Rigid-body free-fall smoke using MuJoCo's Python API.
if importlib.util.find_spec("mujoco") is not None:
    try:
        import mujoco
        xml='<mujoco><worldbody><body pos="0 0 1"><freejoint/><geom type="sphere" size="0.1" mass="1"/></body></worldbody></mujoco>'
        model=mujoco.MjModel.from_xml_string(xml); data=mujoco.MjData(model)
        for _ in range(120): mujoco.mj_step(model,data)
        z=float(data.qpos[2])
        optional_library_results["mujoco"]={"status":"PASS" if 0.0<z<1.0 else "FAIL","final_z":z}
    except Exception as exc: optional_library_results["mujoco"]={"status":"UNAVAILABLE","error":f"{type(exc).__name__}: {exc}"}
else: optional_library_results["mujoco"]={"status":"NOT_INSTALLED"}
print(json.dumps(optional_library_results,indent=2))

## 8. PyAMG optional algebraic multigrid preconditioner

PyAMG is an optional sparse preconditioning backend. When installed, the same sparse operator is solved with a smoothed-aggregation multigrid preconditioner and SciPy CG. This tests a different, established numerical strategy against the same analytical solution.

In [ ]:
pyamg_result = None
if importlib.util.find_spec("pyamg") is not None:
    import pyamg
    A,b,exact,h = poisson_system(64)
    ml = pyamg.smoothed_aggregation_solver(A)
    M = ml.aspreconditioner()
    started = time.perf_counter()
    x,info = cg(A,b,M=M,rtol=1e-11,atol=0.0,maxiter=20*64*64)
    elapsed = time.perf_counter()-started
    err = float(np.linalg.norm(x.reshape(64,64)-exact)/np.linalg.norm(exact))
    residual = float(np.linalg.norm(A@x-b)/np.linalg.norm(b))
    pyamg_result = {"backend":"pyamg","version":pyamg.__version__,"device":"CPU","precision":"float64",
                    "relative_l2_error":err,"relative_residual":residual,"info":int(info),"wall_seconds":elapsed}
    assert info == 0 and err < MAX_RELATIVE_L2_ERROR and residual < MAX_RELATIVE_RESIDUAL, pyamg_result
    print(pyamg_result)
else:
    pyamg_result = {"backend":"pyamg","status":"NOT_INSTALLED"}
    print(pyamg_result)

## 9. Build a platform-bound result envelope

The result binds project, configuration, run, solver, backend, input hash, output hash, validation checks, numerical metrics, runtime, device, precision and resource use. The detached SHA-256 digest is over the canonical JSON payload. The platform's `validateComputeResultAsync()` recomputes the digest before accepting the result. A passing numerical run remains `method=SIMULATION`, `fidelity=UNVERIFIED`, with HIL/L5/physical-verification claims forced false.

Before running this cell, update the three identifiers at the top to real canonical IDs and keep the run ID unique. Do not fabricate requirement IDs for production work.

In [ ]:
def canonical_json(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False, allow_nan=False)

def sha256_json(obj):
    return hashlib.sha256(canonical_json(obj).encode("utf-8")).hexdigest()

# Backend choice is explicit so it can be bound to the platform plan before launch.
# No silent fallback is allowed when the requested backend/device is unavailable.
requested_accelerator_match = False
if REQUESTED_BACKEND_ID == "numpy-scipy":
    requested_accelerator_match = (REQUESTED_ACCELERATOR == "cpu")
    primary_backend = "numpy-scipy"
    primary_device = "CPU"
    primary_version = scipy.__version__
    primary_error = reference_runs[-1]["relative_l2_error"]
    primary_residual = reference_runs[-1]["relative_residual"]
    primary_wall = reference_runs[-1]["wall_seconds"]
elif REQUESTED_BACKEND_ID == "numba":
    requested_accelerator_match = bool(numba_result and "relative_l2_error" in numba_result and REQUESTED_ACCELERATOR == "cpu")
    primary_backend = "numba"
    primary_device = numba_result.get("device", "UNAVAILABLE") if numba_result else "UNAVAILABLE"
    primary_version = numba_result.get("version", "unavailable") if numba_result else "unavailable"
    primary_error = numba_result.get("relative_l2_error", 1.0) if numba_result else 1.0
    primary_residual = numba_result.get("relative_residual", 1.0) if numba_result else 1.0
    primary_wall = numba_result.get("wall_seconds", 0.0) if numba_result else 0.0
elif REQUESTED_BACKEND_ID == "cupy":
    requested_accelerator_match = bool(cupy_result and "relative_l2_error" in cupy_result and REQUESTED_ACCELERATOR == "nvidia-gpu")
    primary_backend = "cupy"
    primary_device = cupy_result.get("device", "UNAVAILABLE") if cupy_result else "UNAVAILABLE"
    primary_version = cupy_result.get("version", "unavailable") if cupy_result else "unavailable"
    primary_error = cupy_result.get("relative_l2_error", 1.0) if cupy_result else 1.0
    primary_residual = cupy_result.get("relative_residual", 1.0) if cupy_result else 1.0
    primary_wall = cupy_result.get("wall_seconds", 0.0) if cupy_result else 0.0
elif REQUESTED_BACKEND_ID == "pyamg":
    requested_accelerator_match = bool(pyamg_result and "relative_l2_error" in pyamg_result and REQUESTED_ACCELERATOR == "cpu")
    primary_backend = "pyamg"
    primary_device = pyamg_result.get("device", "UNAVAILABLE") if pyamg_result else "UNAVAILABLE"
    primary_version = pyamg_result.get("version", "unavailable") if pyamg_result else "unavailable"
    primary_error = pyamg_result.get("relative_l2_error", 1.0) if pyamg_result else 1.0
    primary_residual = pyamg_result.get("relative_residual", 1.0) if pyamg_result else 1.0
    primary_wall = pyamg_result.get("wall_seconds", 0.0) if pyamg_result else 0.0
elif REQUESTED_BACKEND_ID == "pytorch":
    requested_accelerator_match = bool(torch_result and "relative_l2_error" in torch_result and
                                       ((REQUESTED_ACCELERATOR == "cpu" and torch_result["device"] == "cpu") or
                                        (REQUESTED_ACCELERATOR == "nvidia-gpu" and torch_result["device"].startswith("cuda"))))
    primary_backend = "pytorch"
    primary_device = torch_result.get("device", "UNAVAILABLE") if torch_result else "UNAVAILABLE"
    primary_version = backend_probe["torch"]["version"] or "unavailable"
    primary_error = torch_result.get("relative_l2_error", 1.0) if torch_result else 1.0
    primary_residual = torch_result.get("relative_residual", 1.0) if torch_result else 1.0
    primary_wall = torch_result.get("wall_seconds", 0.0) if torch_result else 0.0
elif REQUESTED_BACKEND_ID == "jax":
    requested_accelerator_match = bool(jax_result and "relative_l2_error" in jax_result and jax_result.get("devicePlatform") == {"cpu":"cpu", "nvidia-gpu":"gpu", "tpu":"tpu"}.get(REQUESTED_ACCELERATOR))
    primary_backend = "jax"
    primary_device = jax_result.get("device", "UNAVAILABLE") if jax_result else "UNAVAILABLE"
    primary_version = backend_probe["jax"]["version"] or "unavailable"
    primary_error = jax_result.get("relative_l2_error", 1.0) if jax_result else 1.0
    primary_residual = jax_result.get("relative_residual", 1.0) if jax_result else 1.0
    primary_wall = jax_result.get("wall_seconds", 0.0) if jax_result else 0.0
else:
    raise ValueError("REQUESTED_BACKEND_ID must be numpy-scipy, pytorch, jax, numba, cupy or pyamg")
primary_precision = "float64"
use_gpu = REQUESTED_ACCELERATOR == "nvidia-gpu" and requested_accelerator_match

input_payload = {
    "platform":"Physical AI Platform","iteration":50,"copyright":"Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.",
    "license":"GPL-3.0-only","trademarks":["AyeAI®","AyeAM®","AyeCNSe®"],
    "problem":"2D Dirichlet Poisson manufactured solution",
    "equation":"-laplacian(u)=2*pi^2*sin(pi*x)*sin(pi*y)",
    "exact_solution":"sin(pi*x)*sin(pi*y)",
    "grid_sizes":list(GRID_SIZES), "requirements":REQUIREMENT_IDS,
    "project_id":PROJECT_ID,"configuration_id":CONFIGURATION_ID,"solver_id":"poisson-2d-cg",
    "requested_backend":REQUESTED_BACKEND_ID,"requested_accelerator":REQUESTED_ACCELERATOR
}
input_hash = sha256_json(input_payload)
plan_id = f"{PROJECT_ID}:{CONFIGURATION_ID}:{RUN_ID}"
checks = [
    {"id":"GRID_REFINEMENT_MONOTONIC","passed":all(errors[i+1] < errors[i] for i in range(len(errors)-1))},
    {"id":"GRID_REFINEMENT_ORDER","passed":all(order > 1.7 for order in orders)},
    {"id":"ANALYTICAL_ERROR_BOUND","passed":primary_error <= MAX_RELATIVE_L2_ERROR},
    {"id":"LINEAR_SOLVER_RESIDUAL","passed":primary_residual <= MAX_RELATIVE_RESIDUAL},
    {"id":"REQUESTED_BACKEND_AVAILABLE","passed":requested_accelerator_match},
]
if numba_result and "relative_l2_error" in numba_result:
    checks.append({"id":"NUMBA_STENCIL_VALIDATION","passed":numba_result["relative_l2_error"] <= MAX_RELATIVE_L2_ERROR and numba_result["relative_residual"] <= MAX_RELATIVE_RESIDUAL})
if cupy_result and "relative_l2_error" in cupy_result:
    checks.append({"id":"CUPY_SPARSE_SOLVER_VALIDATION","passed":cupy_result["relative_l2_error"] <= MAX_RELATIVE_L2_ERROR and cupy_result["relative_residual"] <= MAX_RELATIVE_RESIDUAL})
if pyamg_result and "relative_l2_error" in pyamg_result:
    checks.append({"id":"PYAMG_PRECONDITIONED_SOLVER_VALIDATION","passed":pyamg_result["relative_l2_error"] <= MAX_RELATIVE_L2_ERROR and pyamg_result["relative_residual"] <= MAX_RELATIVE_RESIDUAL})
if torch_result is not None and "relative_l2_error" in torch_result:
    checks.append({"id":"PYTORCH_STENCIL_VALIDATION","passed":torch_result["relative_l2_error"] <= MAX_RELATIVE_L2_ERROR and torch_result["relative_residual"] <= MAX_RELATIVE_RESIDUAL})
if jax_result and "relative_l2_error" in jax_result:
    checks.append({"id":"JAX_STENCIL_VALIDATION","passed":jax_result["relative_l2_error"] <= MAX_RELATIVE_L2_ERROR and jax_result["relative_residual"] <= MAX_RELATIVE_RESIDUAL})
if cupy_result and cupy_result.get("array_parity"):
    checks.append({"id":"CUPY_ARRAY_PARITY","passed":cupy_result["array_parity"] == "PASS"})
if taichi_result and taichi_result.get("parity"):
    checks.append({"id":"TAICHI_KERNEL_PARITY","passed":taichi_result["parity"] == "PASS"})
if warp_result and warp_result.get("parity"):
    checks.append({"id":"WARP_KERNEL_PARITY","passed":warp_result["parity"] == "PASS"})
for lib_name, lib_result in optional_library_results.items():
    if lib_result.get("status") in ("PASS", "FAIL"):
        checks.append({"id":f"OPTIONAL_{lib_name.upper()}_SMOKE","passed":lib_result["status"] == "PASS"})

output_payload = {
    "reference_grid_runs":[{k:v for k,v in r.items() if k != "solution"} for r in reference_runs],
    "observed_orders":orders,"torch":torch_result,"jax":jax_result,"numba":numba_result,"cupy":cupy_result,"pyamg":pyamg_result,"taichi":taichi_result,"warp":warp_result,"optional_library_results":optional_library_results,
    "backend_probe":backend_probe
}
output_hash = sha256_json(output_payload)
started = runtime["started_utc"]
finished = datetime.now(timezone.utc).isoformat()
result_payload = {
    "schema":"pap.hosted-compute-result/1.0", "planId":plan_id,
    "projectId":PROJECT_ID,"configurationId":CONFIGURATION_ID,"runId":RUN_ID,
    "solverId":"poisson-2d-cg","backendId":primary_backend,"backendVersion":primary_version,
    "metadata":{"platform":"Physical AI Platform","iteration":50,"license":"GPL-3.0-only",
                "copyright":"Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.",
                "trademarks":["AyeAI®","AyeAM®","AyeCNSe®"],
                "notebook":"notebooks/PAP_Hosted_Compute_Backends.ipynb"},
    "status":"COMPLETED" if all(c["passed"] for c in checks) else "FAILED",
    "runtime":f"Python {runtime['python']} / hosted notebook", "device":primary_device,
    "precision":primary_precision,"inputHash":input_hash,"outputHash":output_hash,
    "startedAt":started,"finishedAt":finished,"executionMode":"COLAB_HOSTED",
    "checks":checks,
    "metrics":{"relativeL2Error":primary_error,"relativeResidual":primary_residual,
               "gridRefinementOrderMin":min(orders),"scipyIterationsFineGrid":reference_runs[-1]["iterations"]},
    "validation":{"passed":all(c["passed"] for c in checks),"error":primary_error,
                   "method":"manufactured-solution-analytical-comparison-and-grid-refinement"},
    "resourceUsage":{"wallSeconds":float(primary_wall),"peakMemoryGiB":float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1024**2)),"gpuCount":1 if use_gpu else 0}
}
digest_payload = canonical_json(result_payload)
result = {**result_payload,"digestPayload":digest_payload,
          "resultDigest":hashlib.sha256(digest_payload.encode("utf-8")).hexdigest()}

print(json.dumps({k:v for k,v in result.items() if k != "digestPayload"},indent=2))


### Resource accounting note

The notebook records Linux `ru_maxrss` as peak Python-process resident set size (host memory). This is a conservative process-level high-water mark, not isolated per-solver allocation. When PyTorch CUDA is used, its allocator high-water mark is also recorded separately as `peakDeviceMemoryGiB`. The platform memory budget applies to the host-process field in this version; a future runner may enforce separate host/device budgets.

In [ ]:
# Capture the CUDA allocator high-water mark where applicable and re-seal the result.
peak_device_memory_gib = None
if REQUESTED_BACKEND_ID == "pytorch" and use_gpu:
    try:
        peak_device_memory_gib = float(torch.cuda.max_memory_allocated() / (1024**3))
    except Exception:
        peak_device_memory_gib = None
elif REQUESTED_BACKEND_ID == "jax" and use_gpu and jax_result:
    peak_device_memory_gib = jax_result.get("peakDeviceMemoryGiB")
elif REQUESTED_BACKEND_ID == "cupy" and use_gpu and cupy_result and "relative_l2_error" in cupy_result:
    try:
        peak_device_memory_gib = float(cp.get_default_memory_pool().total_bytes() / (1024**3))
    except Exception:
        peak_device_memory_gib = None
if peak_device_memory_gib is not None:
    result_payload["resourceUsage"]["peakDeviceMemoryGiB"] = peak_device_memory_gib
# Rebuild detached digest after final resource accounting.
digest_payload = canonical_json(result_payload)
result = {**result_payload,"digestPayload":digest_payload,
          "resultDigest":hashlib.sha256(digest_payload.encode("utf-8")).hexdigest()}
print({"peakHostProcessRssGiB":result_payload["resourceUsage"]["peakMemoryGiB"],
       "peakDeviceMemoryGiB":peak_device_memory_gib,"resourceUsage":result_payload["resourceUsage"]})

## 10. Export and import workflow

The current validator requires measured wall time, measured peak memory, GPU count, passing validation checks and a verified digest. If the runtime cannot provide peak memory, the result correctly remains unaccepted until a trusted runner-side resource sampler is added. This is deliberate fail-closed behavior.

The notebook writes the result envelope and its output bundle to `/content`. Download the JSON from the browser. Grok should wire the platform import flow to `validateComputeResultAsync(result, plan)` and only then convert the returned `evidenceDraft` into the existing canonical Artifact/Evidence binding. Do not write directly to the canonical graph from an untrusted notebook. The notebook should never be permitted to set L5, HIL, field, certification or physical-verification status.

In [ ]:
# Save reproducibility bundle. A real project should additionally include the canonical input JSON and mesh/model artifact.
from pathlib import Path
import zipfile
out_dir = Path("/content/pap-hosted-compute")
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir/"compute-result.json").write_text(json.dumps(result,indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
(out_dir/"compute-input.json").write_text(json.dumps(input_payload,indent=2,ensure_ascii=False),encoding="utf-8")
(out_dir/"compute-output-summary.json").write_text(json.dumps(output_payload,indent=2,ensure_ascii=False,allow_nan=False),encoding="utf-8")
bundle_path = Path("/content/PAP_Hosted_Compute_Results.zip")
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for artifact in out_dir.iterdir():
        zf.write(artifact, arcname=artifact.name)
print("Artifacts:", *[str(x) for x in out_dir.iterdir()], str(bundle_path), sep="\n - ")
try:
    from google.colab import files
    files.download(str(bundle_path))
except Exception:
    print("Not running in Colab: retrieve the result bundle from", bundle_path)


## 11. Acceptance matrix and interpretation

| Test | Expected result | What it establishes |
|---|---|---|
| Analytical solution | Relative L2 error within stated bound | Agreement with this manufactured case |
| Linear solver residual | Below configured bound | Algebraic solve quality, not model validity |
| Grid refinement | Monotone error reduction and observed order > 1.7 | Consistency with second-order stencil on this smooth case |
| PyTorch CPU/GPU | Error and residual within bounds | Same discrete problem on selected tensor device |
| JAX | Error and residual within bounds if available | Optional XLA backend consistency |
| CuPy parity | Array reduction matches NumPy if installed and device usable | Basic array backend correctness only |
| Result digest | SHA-256 over detached canonical payload | Transfer integrity, not authorship or trusted identity |
| Canonical import | `validateComputeResultAsync` returns ACCEPTED | Contract compliance only; not engineering certification |

Boundary policy: invalid/missing project or configuration IDs, mismatched input hashes, non-finite metrics, failed checks, invalid timestamps, budget overruns, missing memory measurements or bad digests must HOLD. At the numerical boundary, the tolerance is inclusive (`error <= tolerance`); exceeding it by any positive amount fails. A result may be numerically accepted and still remain `SIMULATION / UNVERIFIED`.

## 8. Limitations and next integrations

- The current numerical case is a Poisson manufactured-solution benchmark. It does not validate Navier-Stokes, turbulence, compressibility, contact, constitutive models, mesh quality or coupled multiphysics.
- PETSc, FEniCSx, MFEM, SU2, OpenFOAM, Elmer, Chrono and other native tools are catalogued as external/advanced backends; this notebook does not pretend to compile and execute every solver in a generic Colab image.
- Colab GPU/TPU type, availability, quotas, idle timeouts and VM lifetime vary. See <https://research.google.com/colaboratory/faq.html>.
- A GPU device being visible does not prove the kernel is accelerated; inspect the reported device and run timing/throughput benchmarks.
- For long-running, MPI, multi-GPU, large-mesh or regulated workloads, use a dedicated hosted runner with pinned images, driver matrix, resource telemetry and job scheduler. Keep the browser as the control plane.
- A validated numerical result still needs independent engineering review, model verification, validation evidence, uncertainty analysis and domain-specific acceptance criteria.